# Dynamic Parallel Execution with Map-Reduce using Send in LangGraph

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
1. **Understand the Map-Reduce pattern** in the context of LangGraph agentic systems
2. **Master the `Send` construct** for dynamic parallel execution
3. **Build a practical Map-Reduce workflow** that generates reports from parallel Q&A generation
4. **Handle dynamic state distribution** when the number of parallel tasks is not known at compile time

---

## 📚 Introduction to Map-Reduce in LangGraph

Map-reduce operations are essential for efficient task decomposition and parallel processing. This pattern is fundamental in distributed computing and is equally powerful in agentic AI systems.

### The Two Phases of Map-Reduce:

| Phase | Description | Example |
|-------|-------------|---------|
| **Map** | Break a task into smaller sub-tasks, processing each sub-task in parallel | Generate multiple questions, then answer each one in parallel |
| **Reduce** | Aggregate the results across all completed, parallelized sub-tasks | Combine all Q&A pairs into a comprehensive report |

---

## 🎯 What We Will Build

We will design a **Report Generation System** that demonstrates Map-Reduce:

1. **Map Phase**: 
   - Generate a set of questions about a given topic
   - Answer each question **in parallel** using `Send(...)`

2. **Reduce Phase**: 
   - Compile a comprehensive report based on all the Q&A pairs

---

## 🖼️ Architecture Diagram

![Map-Reduce Architecture](https://i.imgur.com/SN7KifO.png)

---

## 🔑 Why Use the Send Construct?

LangGraph's `Send` function is crucial when you need **dynamic parallelization**:

| Feature | Description |
|---------|-------------|
| **Task Decomposition** | Breaks down a large task into smaller, manageable sub-tasks |
| **Parallel Processing** | Executes sub-tasks concurrently, significantly reducing overall processing time |
| **Result Aggregation** | Combines outcomes from all sub-tasks to form a comprehensive response |
| **Dynamic Distribution** | Unlike static edges, `Send` allows runtime-determined parallelization |

### When to Use Send vs Static Edges?

- **Static Edges**: Use when you know the exact number of parallel branches at compile time (e.g., always 3 parallel paths)
- **Send Function**: Use when the number of parallel tasks is determined at runtime (e.g., variable number of questions generated)





| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), Orchestrator-Worker (`Send` / map-reduce). Fan-out size is chosen at runtime, so this notebook lives under `4. Orchestrator_Worker/`, not Parallelization. |

---

## Step 1: Install Required Dependencies

First, we need to install the necessary packages:
- **langchain**: Core framework for building LLM applications
- **langchain-openai**: OpenAI integration for LangChain
- **langchain-community**: Community-contributed integrations
- **langgraph**: Framework for building stateful, multi-actor applications with LLMs

In [ ]:
# ============================================================================
# INSTALLATION COMMANDS (Uncomment and run if packages are not installed)
# ============================================================================
# Note: Run these only once, then restart the kernel if needed

# !pip install langchain==0.3.14           # Core LangChain framework
# !pip install langchain-openai==0.3.0     # OpenAI models integration  
# !pip install langchain-community==0.3.14 # Community integrations
# !pip install langgraph==0.2.64           # LangGraph for building agent workflows

---

## Step 2: Configure API Credentials

We need to set up the OpenAI API key for accessing GPT models. Using `getpass` ensures your API key is not displayed in the notebook output.

In [ ]:
# ============================================================================
# SECURE API KEY INPUT
# ============================================================================
# Import getpass for secure password/API key input
# This prevents the API key from being displayed in notebook output

# from getpass import getpass

# Securely prompt for the OpenAI API key
# The input will be masked for security
# OPENAI_KEY = getpass('Enter Open AI API Key: ')

### Set Environment Variables

Store the API key as an environment variable so LangChain can access it automatically.

In [ ]:
# Set the API key as an environment variable
# LangChain automatically reads OPENAI_API_KEY from environment
# import os

# os.environ['OPENAI_API_KEY'] = OPENAI_KEY

# print("Environment variable set successfully!")

---

## Step 3: Define Agent State Schema

In LangGraph, the **State** is a central concept that holds all the data flowing through the graph. We need to define:

1. **Pydantic Models**: For structured output from the LLM (enforces schema validation)
2. **TypedDict State**: The overall state that flows through the graph

### Key Concept: The `Annotated` Type with `operator.add`

Notice the `Annotated[list, operator.add]` for `answers`. This is a **reducer function** that tells LangGraph how to combine multiple updates to the same field:
- When multiple `generate_answer` nodes run in parallel, each returns an answer
- The `operator.add` reducer **concatenates** all these answers into a single list
- This is essential for the Map-Reduce pattern!

In [ ]:
# ============================================================================
# STATE SCHEMA DEFINITIONS
# ============================================================================
from typing_extensions import TypedDict
from pydantic import BaseModel
import operator
from typing import Annotated

# ----------------------------------------------------------------------------
# PYDANTIC MODELS FOR STRUCTURED LLM OUTPUT
# These models define the expected structure of LLM responses
# Using with_structured_output() ensures the LLM returns data in this format
# ----------------------------------------------------------------------------

class Questions(BaseModel):
    """
    Schema for the question generation node output.
    The LLM will return a list of questions about the given topic.
    """
    questions: list[str]  # List of sub-questions to explore

class Answer(BaseModel):
    """
    Schema for the answer generation node output.
    Each answer includes the original question for context.
    """
    question: str   # The question being answered
    answer: str     # The detailed answer

class Report(BaseModel):
    """
    Schema for the final report compilation.
    Contains the synthesized report text.
    """
    report: str     # The final comprehensive report

# ----------------------------------------------------------------------------
# OVERALL STATE FOR THE LANGGRAPH WORKFLOW
# This TypedDict defines all data that flows through the graph
# ----------------------------------------------------------------------------

class OverallState(TypedDict):
    """
    The main state object that persists throughout the graph execution.
    
    Attributes:
        topic: The input topic for report generation
        questions: List of generated sub-questions (Map phase input)
        answers: Accumulated answers from parallel execution (Map phase output)
                 Note: Uses Annotated[list, operator.add] as a REDUCER
                 This means when multiple nodes update 'answers', 
                 the values are concatenated (added) together!
        report: The final compiled report (Reduce phase output)
    """
    topic: str                              # Input: Topic to research
    questions: list                          # Generated questions (variable count)
    answers: Annotated[list, operator.add]   # KEY: Reducer function for parallel results!
    report: str                              # Output: Final compiled report

print("State schemas defined successfully!")

---

## Step 4: Define Agent Node Functions

Now we define the nodes (functions) that will be executed in our graph. Each node performs a specific task in the Map-Reduce workflow.

### The Four Nodes in Our System:

| Node | Role | Phase |
|------|------|-------|
| `generate_questions` | Creates sub-questions about the topic | Planning |
| `continue_to_answers` | Routes each question to parallel processing using `Send` | Map (Distribution) |
| `generate_answer` | Answers a single question (runs in parallel) | Map (Execution) |
| `compile_report` | Combines all Q&A into a final report | Reduce |

### 🔑 Key Concept: The `Send` Function

The `Send` function is the **magic** that enables dynamic parallel execution:

```python
Send("node_name", {"state_key": "value"})
```

- **First argument**: Name of the target node to execute
- **Second argument**: State/data to pass to that node instance

When `continue_to_answers` returns a **list of `Send` objects**, LangGraph:
1. Creates a separate instance of `generate_answer` for each `Send`
2. Executes all instances **in parallel**
3. Collects all results and merges them using the reducer (`operator.add`)

### Why Send is Different from Static Edges:

| Static Edges | Send Function |
|-------------|---------------|
| Fixed number of parallel paths | Dynamic number of parallel paths |
| Defined at compile time | Determined at runtime |
| Good for known branching | Good for variable iteration |

In [ ]:
# ============================================================================
# LLM SETUP: Initialize the Language Model
# ============================================================================
# We use helper functions to create LLM instances with proper configuration
# These functions handle API key loading and model configuration

import os
import sys

# Add parent directory to path for importing helpers
sys.path.append(os.path.abspath("../.."))

# Import our LLM factory functions
# Available options:
#   - get_groq_llm(): Creates a Groq-hosted LLM (fast inference, open-source models)
#   - get_openai_llm(): Creates an OpenAI GPT model
#   - get_databricks_llm(): Creates a Databricks-hosted LLM
from helpers.utils import get_groq_llm, get_openai_llm, get_databricks_llm

print("LLM helpers imported successfully!")

# ----------------------------------------------------------------------------
# Initialize the LLM
# Choose your preferred LLM provider by uncommenting the appropriate line
# ----------------------------------------------------------------------------

# Option 1: Databricks-hosted Claude (recommended for this demo)
llm = get_databricks_llm("databricks-claude-sonnet-4")

# Option 2: Groq for fast inference with open-source models
# llm = get_groq_llm()

# Option 3: OpenAI GPT models
# llm = get_openai_llm()

# Print confirmation of which LLM is being used
if hasattr(llm, 'model_name'):
    print(f"✅ LLM initialized: {llm.model_name}")
elif hasattr(llm, 'model'):
    print(f"✅ LLM initialized: {llm.model} (Databricks)")
else:
    print("✅ LLM initialized successfully")

In [ ]:
# ============================================================================
# NODE FUNCTION DEFINITIONS
# ============================================================================
# Each function represents a node in our LangGraph workflow.
# Nodes receive state as input and return state updates as output.

from langgraph.constants import Send  # Import the Send construct for dynamic parallel execution

# ============================================================================
# NODE 1: GENERATE QUESTIONS (Planning Phase)
# ============================================================================
# This node takes the input topic and generates a list of sub-questions.
# The number of questions can vary based on topic complexity.

def generate_questions(state: OverallState):
    """
    Generate sub-questions about the given topic.
    
    This is the PLANNING phase of our Map-Reduce workflow.
    The LLM analyzes the topic and generates relevant questions that,
    when answered, will provide comprehensive coverage of the topic.
    
    Args:
        state: Contains the 'topic' to research
        
    Returns:
        Dict with 'questions' list - these will be processed in parallel
    """
    # Prompt engineering: We instruct the LLM to generate a variable number
    # of questions based on topic complexity. This demonstrates how Send
    # handles dynamic parallelization.
    questions_prompt = """Generate a list of concise sub-questions related to this overall topic: {topic}
                          which would help build a good report.
                          Follow these rules for question generation:
                            - Do not create very long questions.
                            - Number of questions should always be 3 for simple topics (Birds, Animals, AI)
                              and 5 for more complex topics (Outlook for ..., Impact of ...)
                       """
    
    prompt = questions_prompt.format(topic=state["topic"])
    
    # Use structured output to ensure the LLM returns a valid Questions object
    response = llm.with_structured_output(Questions).invoke(prompt)
    
    print(f"📝 Generated {len(response.questions)} questions for topic: {state['topic']}")
    
    return {"questions": response.questions}


# ============================================================================
# NODE 2: GENERATE ANSWER (Map Phase - Execution)
# ============================================================================
# This node answers a SINGLE question. Multiple instances run in PARALLEL.

def generate_answer(state: Answer):
    """
    Generate an answer for a single question.
    
    This is the MAP EXECUTION phase. This function is called MULTIPLE TIMES
    in parallel - once for each question generated in the planning phase.
    
    IMPORTANT: Notice the state type is 'Answer', not 'OverallState'.
    This is because Send() passes a custom state to each parallel instance.
    
    Args:
        state: Contains just the 'question' to answer (passed via Send)
        
    Returns:
        Dict with 'answers' list - these get MERGED via operator.add reducer
    """
    answer_prompt = """Generate the answer about {question}."""
    prompt = answer_prompt.format(question=state["question"])
    
    # Get structured answer from LLM
    response = llm.with_structured_output(Answer).invoke(prompt)
    
    # IMPORTANT: Return as a LIST so the reducer (operator.add) can concatenate
    # all parallel results together!
    return {"answers": [{"question": state["question"], "answer": response.answer}]}


# ============================================================================
# NODE 3: CONTINUE TO ANSWERS (Map Phase - Distribution)
# ============================================================================
# This is a CONDITIONAL EDGE function that uses Send for parallel dispatch.

def continue_to_answers(state: OverallState):
    """
    Distribute questions to parallel answer generation using Send.
    
    This is the MAP DISTRIBUTION phase. It's the key function that enables
    dynamic parallel execution in LangGraph.
    
    HOW IT WORKS:
    1. Takes the list of questions from state
    2. Creates a Send object for EACH question
    3. Returns a LIST of Send objects
    4. LangGraph executes all Send targets IN PARALLEL
    
    Args:
        state: Contains 'questions' list from generate_questions
        
    Returns:
        List[Send] - Each Send creates a parallel instance of generate_answer
    """
    # 🔑 THE MAGIC: Create a Send for each question
    # Each Send("generate_answer", {...}) will:
    #   1. Create a new instance of the generate_answer node
    #   2. Pass the question in a custom state
    #   3. Execute in parallel with other Send instances
    
    sends = [Send("generate_answer", {"question": q}) for q in state["questions"]]
    
    print(f"🚀 Dispatching {len(sends)} parallel answer generation tasks...")
    
    return sends


# ============================================================================
# NODE 4: COMPILE REPORT (Reduce Phase)
# ============================================================================
# This node aggregates all parallel results into a final report.

def compile_report(state: OverallState):
    """
    Compile all Q&A pairs into a comprehensive report.
    
    This is the REDUCE phase. By the time this runs:
    - All parallel generate_answer calls have completed
    - All answers have been merged into state['answers'] via the reducer
    - We can now synthesize everything into a final report
    
    Args:
        state: Contains 'topic', 'questions', and all 'answers'
        
    Returns:
        Dict with 'report' - the final synthesized output
    """
    # Format all Q&A pairs for the prompt
    q_and_a = "\n\n".join(
        [f"Q: {qa['question']}\nA: {qa['answer']}" for qa in state["answers"]]
    )
    
    print(f"📊 Compiling report from {len(state['answers'])} Q&A pairs...")
    
    report_prompt = """Below are a bunch of questions and answers about topic:
                       {topic}.
                       Generate a detailed report from this about the topic.
                       {q_and_a}"""
    
    prompt = report_prompt.format(topic=state["topic"], q_and_a=q_and_a)
    
    response = llm.with_structured_output(Report).invoke(prompt)
    
    return {"report": response.report}


print("✅ All node functions defined successfully!")

---

## Step 5: Build the LangGraph Workflow

Now we assemble our nodes into a complete graph. The key insight is how we connect the `generate_questions` node to parallel answer generation using **conditional edges**.

### Graph Structure:

```
START → generate_questions → [PARALLEL: generate_answer × N] → compile_report → END
```

### Key API Components:

| Component | Purpose |
|-----------|---------|
| `StateGraph(OverallState)` | Create a graph with our state schema |
| `add_node(name, fn)` | Register a node function |
| `add_edge(from, to)` | Create a direct edge between nodes |
| `add_conditional_edges(from, fn, targets)` | Create dynamic routing (used with Send) |
| `compile()` | Build the executable agent |

In [ ]:
# ============================================================================
# BUILD THE LANGGRAPH WORKFLOW
# ============================================================================
from langgraph.graph import StateGraph, START, END

# ----------------------------------------------------------------------------
# Step 1: Create the StateGraph with our state schema
# The OverallState TypedDict defines all data that flows through the graph
# ----------------------------------------------------------------------------
graph = StateGraph(OverallState)

# ----------------------------------------------------------------------------
# Step 2: Add all nodes to the graph
# Each node is a function that processes state and returns updates
# ----------------------------------------------------------------------------
graph.add_node("generate_questions", generate_questions)  # Planning phase
graph.add_node("generate_answer", generate_answer)        # Map phase (parallel)
graph.add_node("compile_report", compile_report)          # Reduce phase

# ----------------------------------------------------------------------------
# Step 3: Define the edges (control flow)
# ----------------------------------------------------------------------------

# Edge 1: Start with question generation
graph.add_edge(START, "generate_questions")

# Edge 2: CONDITIONAL EDGE - The key to dynamic parallelization!
# The continue_to_answers function returns a list of Send objects,
# each targeting "generate_answer" with a different question.
# The third argument ["generate_answer"] tells LangGraph which nodes
# might be targeted by the conditional function.
graph.add_conditional_edges(
    "generate_questions",      # From this node...
    continue_to_answers,       # ...run this function to decide next steps
    ["generate_answer"]        # ...which may route to these nodes
)

# Edge 3: After all parallel answer generations complete, compile the report
# This edge is triggered once ALL Send targets have completed
graph.add_edge("generate_answer", "compile_report")

# Edge 4: End after report compilation
graph.add_edge("compile_report", END)

# ----------------------------------------------------------------------------
# Step 4: Compile the graph into an executable agent
# ----------------------------------------------------------------------------
agent = graph.compile()

print("✅ Agent graph compiled successfully!")

In [ ]:
# ============================================================================
# VISUALIZE THE GRAPH STRUCTURE
# ============================================================================
# LangGraph provides built-in visualization using Mermaid diagrams.
# This helps understand the workflow structure at a glance.
#
# In the visualization, you'll see:
#   - START → generate_questions: Entry point
#   - generate_questions → generate_answer: Conditional edge (Send-based)
#   - generate_answer → compile_report: Aggregation point
#   - compile_report → END: Final output

from IPython.display import display, Image

# Render the graph as a PNG image using Mermaid
Image(agent.get_graph().draw_mermaid_png())

---

## Step 6: Run and Test the Agent

Let's test our Map-Reduce agent with different topics to see how it handles:
1. **Simple topics** → Generates fewer questions (3)
2. **Complex topics** → Generates more questions (5)

### Understanding the Output:

When we stream the agent execution, we see state updates at each step:
1. `generate_questions`: Shows the list of generated questions
2. `generate_answer`: Shows each Q&A pair (may appear in parallel)
3. `compile_report`: Shows the final synthesized report

### Test Cases:

| Topic | Expected Complexity | Expected Questions |
|-------|--------------------|--------------------|
| "Artificial Intelligence" | Simple | ~3 questions |
| "Animals" | Simple | ~3 questions |
| "Impact of AI on jobs" | Complex | ~5 questions |

In [ ]:
# ============================================================================
# TEST 1: Simple Topic - "Artificial Intelligence"
# ============================================================================
# Expected behavior:
#   - generate_questions creates ~3 sub-questions
#   - Each question is answered in parallel via Send
#   - All answers are aggregated and compiled into a report

from IPython.display import display, Markdown

print("=" * 70)
print("🧪 TEST 1: Simple Topic - Artificial Intelligence")
print("=" * 70)

# Stream the agent execution to observe each step
# The stream yields state updates after each node completes
for state in agent.stream({"topic": "Artificial Intelligence"}):
    print(state)
    print("-" * 50)
    
    # Display the final report in formatted Markdown
    if 'compile_report' in state:
        print("\n📋 FINAL REPORT:")
        display(Markdown(state['compile_report']['report']))

In [ ]:
# ============================================================================
# TEST 2: Simple Topic - "Animals"
# ============================================================================
# Another simple topic to verify consistent behavior with few questions

print("\n" + "=" * 70)
print("🧪 TEST 2: Simple Topic - Animals")
print("=" * 70)

for state in agent.stream({"topic": "Animals"}):
    print(state)
    print("-" * 50)
    
    if 'compile_report' in state:
        print("\n📋 FINAL REPORT:")
        display(Markdown(state['compile_report']['report']))

In [ ]:
# ============================================================================
# TEST 3: Complex Topic - "Impact of AI on jobs"
# ============================================================================
# A more complex topic should generate more questions (~5),
# demonstrating how Send handles variable parallelization

print("\n" + "=" * 70)
print("🧪 TEST 3: Complex Topic - Impact of AI on jobs")
print("=" * 70)
print("Note: Expect more questions for this complex topic!\n")

for state in agent.stream({"topic": "Impact of AI on jobs"}):
    print(state)
    print("-" * 50)
    
    if 'compile_report' in state:
        print("\n📋 FINAL REPORT:")
        display(Markdown(state['compile_report']['report']))

## 🎓 Summary and Key Takeaways

Congratulations! You've learned how to implement the **Map-Reduce pattern** in LangGraph using the **Send construct** for dynamic parallel execution.

### Key Concepts Covered:

| Concept | Description |
|---------|-------------|
| **Map-Reduce Pattern** | Decompose tasks → Process in parallel → Aggregate results |
| **Send Construct** | `Send("node_name", state)` - Dynamically dispatch to parallel nodes |
| **State Reducers** | `Annotated[list, operator.add]` - Merge parallel results automatically |
| **Conditional Edges** | Route dynamically based on runtime conditions |

### When to Use This Pattern:

✅ **Good Use Cases:**
- Report generation from multiple sub-queries
- Research assistants that explore topics from multiple angles
- Data processing with variable number of items
- Any workflow where parallelization count is runtime-determined

❌ **Avoid When:**
- Fixed number of parallel tasks (use static edges instead)
- Sequential dependencies between tasks
- Simple single-step operations

### Architecture Recap:

```
┌─────────────┐     ┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│    START    │────▶│   generate_     │────▶│ continue_to_    │────▶│   generate_     │
│             │     │   questions     │     │   answers       │     │   answer (×N)   │
└─────────────┘     └─────────────────┘     └─────────────────┘     └────────┬────────┘
                                                   │                         │
                                                   │                         │ (parallel)
                                                   │                         │
                                            ┌──────▼─────────────────────────▼──────┐
                                            │              compile_report           │
                                            └──────────────────┬────────────────────┘
                                                               │
                                                               ▼
                                                          ┌─────────┐
                                                          │   END   │
                                                          └─────────┘
```

### Next Steps:

1. **Experiment**: Try different topics and observe the variable question counts
2. **Extend**: Add error handling for failed LLM calls
3. **Optimize**: Implement caching for repeated questions
4. **Scale**: Apply this pattern to more complex multi-stage workflows

---

## 📚 Additional Resources

- [LangGraph Documentation](https://python.langchain.com/docs/langgraph)
- [Send API Reference](https://langchain-ai.github.io/langgraph/reference/graphs/)
- [Map-Reduce Patterns in Distributed Systems](https://en.wikipedia.org/wiki/MapReduce)